# Truth Through Debate v2
**Multi-Agent LLM Fact-Checking** — 5 free Groq models, hybrid retrieval, Platt calibration

**Before running:**
1. Click the 🔑 key icon in the left sidebar
2. Add secret: Name = `GROQ_API_KEY`, Value = your key from [console.groq.com](https://console.groq.com)
3. Toggle **Notebook access ON**
4. (Optional) Add `TAVILY_API_KEY` from [tavily.com](https://tavily.com) for web search

In [ ]:
# Cell 1 — Install
!pip install groq aiohttp faiss-cpu sentence-transformers rank-bm25 \
             datasets scikit-learn pandas matplotlib tqdm rich \
             python-dotenv requests gradio tavily-python nest-asyncio -q

In [ ]:
# Cell 2 — Upload project zip
from google.colab import files
uploaded = files.upload()  # upload truth_through_debate_v2.zip

import zipfile, sys
with zipfile.ZipFile(list(uploaded.keys())[0]) as z:
    z.extractall('.')
sys.path.insert(0, '/content/ttd_v2')
print('Project loaded.')

In [ ]:
# Cell 3 — API keys
import os, nest_asyncio
nest_asyncio.apply()
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
try:
    os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')
except:
    print('TAVILY_API_KEY not set — web search disabled.')

from truth_through_debate.config import Config
cfg = Config.from_env()
print('Models:')
print(f'  Baseline  : {cfg.baseline_model}')
print(f'  Debater A : {cfg.debater_a_model}')
print(f'  Debater B : {cfg.debater_b_model}')
print(f'  Devil     : {cfg.devils_model}')
print(f'  Judge     : {cfg.judge_model}')
print(f'  Scorer    : {cfg.scorer_model}')

In [ ]:
# Cell 4 — Single claim demo
from truth_through_debate.pipeline import DebatePipeline

CLAIM = 'The Great Wall of China is visible from space with the naked eye.'

async with DebatePipeline(cfg) as p:
    result = await p.run(CLAIM, ground_truth='FALSE', verbose=True)

print(f'\nVerdict    : {result.verdict}')
print(f'Confidence : {result.confidence:.2f}')
print(f'Reasoning  : {result.reasoning_score}/5')
print(f'Correct    : {result.correct}')
print(f'Explanation: {result.explanation}')

In [ ]:
# Cell 5 — Full debate transcript
for rd in result.rounds:
    print(f'\n{"="*60}')
    print(f'ROUND {rd.round_num}')
    print(f'{"="*60}')
    print(f'[Debater A — TRUE]\n{rd.argument_a}')
    print(f'\n[Debater B — FALSE]\n{rd.argument_b}')
    print(f'\n[Devil\'s Advocate]\n{rd.devils_argument}')

print(f'\nEvidence retrieved ({len(result.evidence)} snippets):')
for i, ev in enumerate(result.evidence, 1):
    print(f'  [{i}] ({ev.source}) {ev.title}: {ev.text[:120]}...')

In [ ]:
# Cell 6 — Batch evaluation (built-in 50 claims, no download needed)
from truth_through_debate.evaluation.evaluator import run_evaluation
from truth_through_debate.evaluation.fever_loader import load_builtin

claims = load_builtin(n=20)  # increase to 50 for full run (~30 min)
print(f'Running {len(claims)} claims...')

results, metrics = await run_evaluation(claims, cfg, output_dir='outputs/', plot_calibration=True)

In [ ]:
# Cell 7 — Results table
print(f'\n{"Metric":<30} {"Baseline":>12} {"Debate":>12} {"Delta":>10}')
print('-' * 66)
for m, b, d, delta in metrics.summary_rows():
    print(f'{m:<30} {b:>12} {d:>12} {delta:>10}')

In [ ]:
# Cell 8 — Reliability diagram
from truth_through_debate.calibration.platt import CalibrationPipeline
import matplotlib.pyplot as plt

cal = CalibrationPipeline()
cal.fit(results)
cal.plot_reliability_diagram('outputs/reliability.png')
plt.show()
print(f'ECE before: {cal.ece_before:.4f}  after: {cal.ece_after:.4f}')

In [ ]:
# Cell 9 — Export CSV
import pandas as pd
df = pd.DataFrame([r.to_dict() for r in results])
df.to_csv('outputs/results.csv', index=False)
print(df[['claim','verdict','confidence','calibrated_confidence',
          'reasoning_score','correct','baseline_verdict','baseline_correct']].to_string())

In [ ]:
# Cell 10 — Download outputs
from google.colab import files
files.download('outputs/results.csv')
files.download('outputs/reliability.png')

In [ ]:
# Cell 11 — Launch Gradio UI (optional)
from truth_through_debate.ui.app import launch
launch(share=True)  # prints a public gradio.live URL